# Lambda=0 실험 결과 시각화

**데이터**: `results/combined/consolidated_lambda0_final.csv`  
**그리드**: 17 메서드 × 10 K × 10 TopN × 10 fold = 17,000 (optimized) + 17,000 (baseline)  

| 그래프 | 설명 |
|--------|------|
| Graph 1 | fold별 최적 alpha (method×line) |
| Graph 2 | 유사도별 validation/test RMSE·MAD |
| Graph 3 | 유사도별 validation/test Recall·Precision |
| Graph 4 | alpha 최적화 곡선 (acos/jmsd/msd만 지원) |
| Graph 5 | baseline vs optimized 비교 - RMSE·MAD |
| Graph 6 | baseline vs optimized 비교 - Recall·Precision |

In [8]:
import os
import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
try:
    import plotly.graph_objects as go
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "-q"])
    import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── 경로 설정 ────────────────────────────────────────────────────────────────
BASE       = os.getcwd()  # 노트북은 os.getcwd() 사용
CSV_PATH   = os.path.join(BASE, 'results', 'combined', 'consolidated_lambda0_final.csv')
AHIST_PATH = os.path.join(BASE, 'results', 'combined', 'consolidated_alpha_history_lambda0.csv')

# ── 데이터 로딩 ──────────────────────────────────────────────────────────────
print('Loading consolidated data...')
df = pd.read_csv(CSV_PATH)
df['fold'] = df['fold'].astype(int)
df['K']    = df['K'].astype(int)
df['TopN'] = df['TopN'].astype(int)
print(f'  {len(df):,}행 × {df.shape[1]}열  ({CSV_PATH})')

# alpha history (Graph 4용, 17개 메서드 전체)
ahist = None
if os.path.exists(AHIST_PATH):
    print('Loading alpha history...')
    ahist = pd.read_csv(AHIST_PATH)
    ahist['fold'] = ahist['fold'].astype(int)
    ahist['K']    = ahist['K'].astype(int)
    ahist['TopN'] = ahist['TopN'].astype(int)
    print(f'  {len(ahist):,}행  ({AHIST_PATH})')
    print(f'  메서드({ahist["method"].nunique()}): {sorted(ahist["method"].unique())}')
else:
    print(f'  alpha history 파일 없음: {AHIST_PATH}')

# ── 공통 상수 ────────────────────────────────────────────────────────────────
K_VALUES    = sorted(df['K'].unique())
TOPN_VALUES = sorted(df['TopN'].unique())
FOLD_VALUES = sorted(df['fold'].unique())
METHODS     = sorted(df['method'].unique())
METHODS_3   = ['acos', 'jmsd', 'msd']

# Plotly 색상 팔레트 (17색)
COLORS = px.colors.qualitative.Alphabet[:17]
METHOD_COLOR = {m: COLORS[i] for i, m in enumerate(METHODS)}

print(f'\n메서드 ({len(METHODS)}): {METHODS}')
print(f'K: {K_VALUES}')
print(f'TopN: {TOPN_VALUES}')
print(f'Fold: {FOLD_VALUES}')
print('\n✓ 데이터 로딩 완료')


Loading consolidated data...
  34,000행 × 13열  (d:\Dropbox\python_workspace(백업)\imputation_project\results\combined\consolidated_lambda0_final.csv)
Loading alpha history...
  675,300행  (d:\Dropbox\python_workspace(백업)\imputation_project\results\combined\consolidated_alpha_history_lambda0.csv)
  메서드(17): ['acos', 'ami', 'ari', 'chebyshev', 'cosine', 'cpcc', 'euclidean', 'ipwr', 'itr', 'jaccard', 'jmsd', 'kendall_tau_b', 'manhattan', 'msd', 'pcc', 'spcc', 'src']

메서드 (17): ['acos', 'ami', 'ari', 'chebyshev', 'cosine', 'cpcc', 'euclidean', 'ipwr', 'itr', 'jaccard', 'jmsd', 'kendall_tau_b', 'manhattan', 'msd', 'pcc', 'spcc', 'src']
K: [np.int64(10), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(60), np.int64(70), np.int64(80), np.int64(90), np.int64(100)]
TopN: [np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(25), np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50)]
Fold: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np

---
## Graph 1 — fold별 최적 alpha (method별 선)
**컨트롤**: K, TopN 선택  
**그래프**: x=fold, y=최적 alpha, 각 method가 선으로 표시

In [9]:
out1 = widgets.Output()

w1_k    = widgets.Dropdown(options=K_VALUES,    value=K_VALUES[0],    description='K:',    style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w1_topn = widgets.Dropdown(options=TOPN_VALUES, value=TOPN_VALUES[0], description='TopN:', style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w1_methods = widgets.SelectMultiple(
    options=METHODS, value=METHODS, description='Methods:',
    rows=8, style={'description_width':'60px'}, layout=widgets.Layout(width='220px'))

def plot_graph1(change=None):
    K    = w1_k.value
    TopN = w1_topn.value
    sel_methods = list(w1_methods.value)

    sub = df[(df['type'] == 'optimized') &
             (df['K'] == K) &
             (df['TopN'] == TopN) &
             (df['method'].isin(sel_methods))].copy()

    fig = go.Figure()
    for method in sel_methods:
        m_df = sub[sub['method'] == method].sort_values('fold')
        if m_df.empty:
            continue
        fig.add_trace(go.Scatter(
            x=m_df['fold'], y=m_df['alpha'],
            mode='lines+markers', name=method,
            line=dict(color=METHOD_COLOR[method], width=2),
            marker=dict(size=7),
            hovertemplate=f'<b>{method}</b><br>fold=%{{x}}<br>alpha=%{{y:.4f}}<extra></extra>'
        ))

    fig.update_layout(
        title=f'최적 Alpha by Fold  [K={K}, TopN={TopN}]',
        xaxis=dict(title='Fold', tickvals=FOLD_VALUES),
        yaxis=dict(title='최적 Alpha'),
        legend=dict(title='Method', orientation='v'),
        width=900, height=500,
        hovermode='x unified'
    )

    with out1:
        clear_output(wait=True)
        fig.show()

for w in [w1_k, w1_topn, w1_methods]:
    w.observe(plot_graph1, names='value')

controls1 = widgets.HBox([w1_k, w1_topn, w1_methods])
display(controls1, out1)
plot_graph1()

Output()

---
## Graph 2 — 유사도별 Validation/Test RMSE·MAD
**컨트롤**: K, TopN, fold, metric(RMSE/MAD)  
**그래프**: x=method, y=metric값, validation과 test를 나란히 비교  
*(baseline의 validation은 항상 NaN → test만 표시)*

In [10]:
out2 = widgets.Output()

w2_k      = widgets.Dropdown(options=K_VALUES,    value=K_VALUES[0],    description='K:',    style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w2_topn   = widgets.Dropdown(options=TOPN_VALUES, value=TOPN_VALUES[0], description='TopN:', style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w2_fold   = widgets.Dropdown(options=FOLD_VALUES, value=FOLD_VALUES[0], description='Fold:', style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w2_metric = widgets.RadioButtons(options=['RMSE', 'MAD'], value='RMSE', description='Metric:', style={'description_width':'55px'})

METRIC_MAP_2 = {'RMSE': ('validation_rmse', 'test_RMSE'), 'MAD': (None, 'test_MAD')}

def plot_graph2(change=None):
    K      = w2_k.value
    TopN   = w2_topn.value
    fold   = w2_fold.value
    metric = w2_metric.value

    sub = df[(df['K'] == K) & (df['TopN'] == TopN) & (df['fold'] == fold)].copy()
    opt_df  = sub[sub['type'] == 'optimized'].set_index('method')
    base_df = sub[sub['type'] == 'baseline'].set_index('method')

    val_col, test_col = METRIC_MAP_2[metric]
    methods_sorted = sorted(METHODS)

    fig = go.Figure()

    # Optimized - validation
    if val_col and val_col in opt_df.columns:
        y_val = [opt_df.loc[m, val_col] if m in opt_df.index else np.nan for m in methods_sorted]
        fig.add_trace(go.Bar(
            name=f'Optimized Val {metric}', x=methods_sorted, y=y_val,
            marker_color='steelblue', opacity=0.85,
            hovertemplate='%{x}<br>Val %{y:.5f}<extra>Opt Val</extra>'
        ))

    # Optimized - test
    y_test_opt = [opt_df.loc[m, test_col] if m in opt_df.index else np.nan for m in methods_sorted]
    fig.add_trace(go.Bar(
        name=f'Optimized Test {metric}', x=methods_sorted, y=y_test_opt,
        marker_color='royalblue',
        hovertemplate='%{x}<br>Test %{y:.5f}<extra>Opt Test</extra>'
    ))

    # Baseline - test
    y_test_base = [base_df.loc[m, test_col] if m in base_df.index else np.nan for m in methods_sorted]
    fig.add_trace(go.Bar(
        name=f'Baseline Test {metric}', x=methods_sorted, y=y_test_base,
        marker_color='lightcoral',
        hovertemplate='%{x}<br>Test %{y:.5f}<extra>Base Test</extra>'
    ))

    fig.update_layout(
        title=f'{metric} by Method  [K={K}, TopN={TopN}, Fold={fold}]',
        xaxis=dict(title='Method', tickangle=-35),
        yaxis=dict(title=metric),
        barmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        width=1000, height=500
    )

    with out2:
        clear_output(wait=True)
        fig.show()

for w in [w2_k, w2_topn, w2_fold, w2_metric]:
    w.observe(plot_graph2, names='value')

controls2 = widgets.HBox([w2_k, w2_topn, w2_fold, w2_metric])
display(controls2, out2)
plot_graph2()

Output()

---
## Graph 3 — 유사도별 Validation/Test Recall·Precision
**컨트롤**: K, TopN, fold, metric(Recall/Precision)

In [11]:
out3 = widgets.Output()

w3_k      = widgets.Dropdown(options=K_VALUES,    value=K_VALUES[0],    description='K:',    style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w3_topn   = widgets.Dropdown(options=TOPN_VALUES, value=TOPN_VALUES[0], description='TopN:', style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w3_fold   = widgets.Dropdown(options=FOLD_VALUES, value=FOLD_VALUES[0], description='Fold:', style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w3_metric = widgets.RadioButtons(options=['Recall', 'Precision'], value='Recall', description='Metric:', style={'description_width':'55px'})

METRIC_MAP_3 = {'Recall': ('validation_recall', 'test_Recall'), 'Precision': ('validation_precision', 'test_Precision')}

def plot_graph3(change=None):
    K      = w3_k.value
    TopN   = w3_topn.value
    fold   = w3_fold.value
    metric = w3_metric.value

    sub = df[(df['K'] == K) & (df['TopN'] == TopN) & (df['fold'] == fold)].copy()
    opt_df  = sub[sub['type'] == 'optimized'].set_index('method')
    base_df = sub[sub['type'] == 'baseline'].set_index('method')

    val_col, test_col = METRIC_MAP_3[metric]
    methods_sorted = sorted(METHODS)

    fig = go.Figure()

    # Optimized - validation
    if val_col in opt_df.columns:
        y_val = [opt_df.loc[m, val_col] if m in opt_df.index else np.nan for m in methods_sorted]
        fig.add_trace(go.Bar(
            name=f'Optimized Val {metric}', x=methods_sorted, y=y_val,
            marker_color='mediumseagreen', opacity=0.85,
            hovertemplate='%{x}<br>Val %{y:.5f}<extra>Opt Val</extra>'
        ))

    y_test_opt = [opt_df.loc[m, test_col] if m in opt_df.index else np.nan for m in methods_sorted]
    fig.add_trace(go.Bar(
        name=f'Optimized Test {metric}', x=methods_sorted, y=y_test_opt,
        marker_color='seagreen',
        hovertemplate='%{x}<br>Test %{y:.5f}<extra>Opt Test</extra>'
    ))

    y_test_base = [base_df.loc[m, test_col] if m in base_df.index else np.nan for m in methods_sorted]
    fig.add_trace(go.Bar(
        name=f'Baseline Test {metric}', x=methods_sorted, y=y_test_base,
        marker_color='salmon',
        hovertemplate='%{x}<br>Test %{y:.5f}<extra>Base Test</extra>'
    ))

    fig.update_layout(
        title=f'{metric} by Method  [K={K}, TopN={TopN}, Fold={fold}]',
        xaxis=dict(title='Method', tickangle=-35),
        yaxis=dict(title=metric),
        barmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        width=1000, height=500
    )

    with out3:
        clear_output(wait=True)
        fig.show()

for w in [w3_k, w3_topn, w3_fold, w3_metric]:
    w.observe(plot_graph3, names='value')

controls3 = widgets.HBox([w3_k, w3_topn, w3_fold, w3_metric])
display(controls3, out3)
plot_graph3()

Output()

---
## Graph 4 — Alpha 최적화 곡선 (최적 alpha 표시)
**컨트롤**: K, TopN, fold, method  
**그래프**: x=alpha, y=Validation RMSE, 최적 alpha를 수직선으로 표시  
**메서드**: 17개 전체 (acos/jmsd/msd는 lambda=0 직접 실험; 14개는 lambda=0.002 실험의 raw mse 사용)  


In [12]:
out4 = widgets.Output()

w4_k      = widgets.Dropdown(options=K_VALUES,    value=K_VALUES[0],    description='K:',     style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w4_topn   = widgets.Dropdown(options=TOPN_VALUES, value=TOPN_VALUES[0], description='TopN:',  style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w4_fold   = widgets.Dropdown(options=FOLD_VALUES, value=FOLD_VALUES[0], description='Fold:',  style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w4_method = widgets.Dropdown(options=METHODS,     value=METHODS[0],     description='Method:',style={'description_width':'55px'}, layout=widgets.Layout(width='180px'))
w4_phase  = widgets.RadioButtons(options=['all', 'coarse', 'fine'], value='all', description='Phase:', style={'description_width':'50px'})

def plot_graph4(change=None):
    K      = w4_k.value
    TopN   = w4_topn.value
    fold   = w4_fold.value
    method = w4_method.value
    phase  = w4_phase.value

    with out4:
        clear_output(wait=True)

        if ahist is None:
            print('alpha history 파일을 로딩하지 못했습니다.')
            return

        sub = ahist[(ahist['fold'] == fold) &
                    (ahist['K'] == K) &
                    (ahist['TopN'] == TopN) &
                    (ahist['method'] == method)].copy()

        if sub.empty:
            print(f'데이터 없음: fold={fold}, K={K}, TopN={TopN}, method={method}')
            return

        if phase != 'all':
            sub = sub[sub['phase'] == phase]

        sub = sub.sort_values('alpha')

        # 최적 alpha (consolidated csv에서)
        opt_row = df[(df['fold'] == fold) & (df['K'] == K) &
                     (df['TopN'] == TopN) & (df['method'] == method) &
                     (df['type'] == 'optimized')]
        opt_alpha = float(opt_row['alpha'].values[0]) if len(opt_row) > 0 else None

        fig = go.Figure()

        # Coarse / Fine 별 색
        for ph, color, sym in [('coarse', 'lightsteelblue', 'circle'), ('fine', 'royalblue', 'diamond')]:
            ph_df = sub[sub['phase'] == ph] if 'phase' in sub.columns else sub
            if ph_df.empty:
                continue
            fig.add_trace(go.Scatter(
                x=ph_df['alpha'], y=ph_df['validation_rmse'],
                mode='markers', name=f'{ph}',
                marker=dict(color=color, symbol=sym, size=8),
                hovertemplate='alpha=%{x:.4f}<br>RMSE=%{y:.6f}<extra>' + ph + '</extra>'
            ))

        # 최적 alpha 수직선
        if opt_alpha is not None:
            opt_rmse = sub[np.isclose(sub['alpha'], opt_alpha, atol=1e-4)]['validation_rmse']
            opt_rmse_val = float(opt_rmse.min()) if len(opt_rmse) > 0 else None
            fig.add_vline(x=opt_alpha, line_dash='dash', line_color='red', line_width=2,
                          annotation_text=f'α*={opt_alpha:.4f}', annotation_position='top right')
            if opt_rmse_val:
                fig.add_trace(go.Scatter(
                    x=[opt_alpha], y=[opt_rmse_val], mode='markers',
                    name='최적 alpha', marker=dict(color='red', size=14, symbol='star'),
                    hovertemplate=f'α*={opt_alpha:.4f}<br>RMSE={opt_rmse_val:.6f}<extra>최적</extra>'
                ))

        fig.update_layout(
            title=f'Alpha 최적화 곡선  [{method}, K={K}, TopN={TopN}, Fold={fold}]',
            xaxis=dict(title='Alpha'),
            yaxis=dict(title='Validation RMSE'),
            legend=dict(orientation='h', yanchor='bottom', y=1.02),
            width=900, height=500
        )
        fig.show()

for w in [w4_k, w4_topn, w4_fold, w4_method, w4_phase]:
    w.observe(plot_graph4, names='value')

controls4 = widgets.HBox([w4_k, w4_topn, w4_fold, w4_method, w4_phase])
display(controls4, out4)
plot_graph4()


Output()

---
## Graph 5 — Baseline vs Optimized 비교: RMSE·MAD
**컨트롤**: K, TopN, fold, metric  
**그래프**: x=method, 각 method에 baseline(alpha=1)과 optimized(최적alpha) 두 bar

In [13]:
out5 = widgets.Output()

w5_k      = widgets.Dropdown(options=K_VALUES,    value=K_VALUES[0],    description='K:',    style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w5_topn   = widgets.Dropdown(options=TOPN_VALUES, value=TOPN_VALUES[0], description='TopN:', style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w5_fold   = widgets.Dropdown(options=['All'] + FOLD_VALUES, value='All', description='Fold:', style={'description_width':'40px'}, layout=widgets.Layout(width='160px'))
w5_metric = widgets.RadioButtons(options=['RMSE', 'MAD'], value='RMSE', description='Metric:', style={'description_width':'55px'})

TEST_COL_5 = {'RMSE': 'test_RMSE', 'MAD': 'test_MAD'}

def plot_graph5(change=None):
    K      = w5_k.value
    TopN   = w5_topn.value
    fold   = w5_fold.value
    metric = w5_metric.value
    test_col = TEST_COL_5[metric]

    sub = df[(df['K'] == K) & (df['TopN'] == TopN)].copy()
    if fold != 'All':
        sub = sub[sub['fold'] == fold]

    # fold='All'이면 fold 평균
    if fold == 'All':
        grp = sub.groupby(['method', 'type'])[test_col].mean().reset_index()
        title_suffix = '(fold 평균)'
    else:
        grp = sub.groupby(['method', 'type'])[test_col].mean().reset_index()
        title_suffix = f'Fold={fold}'

    opt_df  = grp[grp['type'] == 'optimized'].set_index('method')
    base_df = grp[grp['type'] == 'baseline'].set_index('method')
    methods_sorted = sorted(METHODS)

    y_opt  = [opt_df.loc[m, test_col]  if m in opt_df.index  else np.nan for m in methods_sorted]
    y_base = [base_df.loc[m, test_col] if m in base_df.index else np.nan for m in methods_sorted]

    # 개선률 (baseline - optimized) / baseline
    improvement = [(b - o) / b * 100 if (b and b != 0) else np.nan
                   for o, b in zip(y_opt, y_base)]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name=f'Optimized (최적α)', x=methods_sorted, y=y_opt,
        marker_color='royalblue',
        hovertemplate='%{x}<br>Opt %{y:.5f}<extra>Optimized</extra>'
    ))
    fig.add_trace(go.Bar(
        name=f'Baseline (α=1)', x=methods_sorted, y=y_base,
        marker_color='lightcoral',
        hovertemplate='%{x}<br>Base %{y:.5f}<extra>Baseline</extra>'
    ))

    # 개선률 텍스트
    for i, (m, imp) in enumerate(zip(methods_sorted, improvement)):
        if not np.isnan(imp):
            fig.add_annotation(
                x=m, y=max(y_opt[i] or 0, y_base[i] or 0),
                text=f'{imp:+.1f}%', showarrow=False,
                yshift=8, font=dict(size=9, color='darkgreen' if imp > 0 else 'red')
            )

    fig.update_layout(
        title=f'Baseline vs Optimized — {metric}  [K={K}, TopN={TopN}, {title_suffix}]',
        xaxis=dict(title='Method', tickangle=-35),
        yaxis=dict(title=f'Test {metric}'),
        barmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        width=1100, height=520
    )

    with out5:
        clear_output(wait=True)
        fig.show()

for w in [w5_k, w5_topn, w5_fold, w5_metric]:
    w.observe(plot_graph5, names='value')

controls5 = widgets.HBox([w5_k, w5_topn, w5_fold, w5_metric])
display(controls5, out5)
plot_graph5()

Output()

---
## Graph 6 — Baseline vs Optimized 비교: Recall·Precision
**컨트롤**: K, TopN, fold, metric

In [14]:
out6 = widgets.Output()

w6_k      = widgets.Dropdown(options=K_VALUES,    value=K_VALUES[0],    description='K:',    style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w6_topn   = widgets.Dropdown(options=TOPN_VALUES, value=TOPN_VALUES[0], description='TopN:', style={'description_width':'40px'}, layout=widgets.Layout(width='150px'))
w6_fold   = widgets.Dropdown(options=['All'] + FOLD_VALUES, value='All', description='Fold:', style={'description_width':'40px'}, layout=widgets.Layout(width='160px'))
w6_metric = widgets.RadioButtons(options=['Recall', 'Precision'], value='Recall', description='Metric:', style={'description_width':'55px'})

TEST_COL_6 = {'Recall': 'test_Recall', 'Precision': 'test_Precision'}

def plot_graph6(change=None):
    K      = w6_k.value
    TopN   = w6_topn.value
    fold   = w6_fold.value
    metric = w6_metric.value
    test_col = TEST_COL_6[metric]

    sub = df[(df['K'] == K) & (df['TopN'] == TopN)].copy()
    if fold != 'All':
        sub = sub[sub['fold'] == fold]

    if fold == 'All':
        grp = sub.groupby(['method', 'type'])[test_col].mean().reset_index()
        title_suffix = '(fold 평균)'
    else:
        grp = sub.groupby(['method', 'type'])[test_col].mean().reset_index()
        title_suffix = f'Fold={fold}'

    opt_df  = grp[grp['type'] == 'optimized'].set_index('method')
    base_df = grp[grp['type'] == 'baseline'].set_index('method')
    methods_sorted = sorted(METHODS)

    y_opt  = [opt_df.loc[m, test_col]  if m in opt_df.index  else np.nan for m in methods_sorted]
    y_base = [base_df.loc[m, test_col] if m in base_df.index else np.nan for m in methods_sorted]

    improvement = [(o - b) / b * 100 if (b and b != 0) else np.nan
                   for o, b in zip(y_opt, y_base)]  # recall/precision: optimized > baseline이 좋음

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name=f'Optimized (최적α)', x=methods_sorted, y=y_opt,
        marker_color='seagreen',
        hovertemplate='%{x}<br>Opt %{y:.5f}<extra>Optimized</extra>'
    ))
    fig.add_trace(go.Bar(
        name=f'Baseline (α=1)', x=methods_sorted, y=y_base,
        marker_color='salmon',
        hovertemplate='%{x}<br>Base %{y:.5f}<extra>Baseline</extra>'
    ))

    for i, (m, imp) in enumerate(zip(methods_sorted, improvement)):
        if not np.isnan(imp):
            fig.add_annotation(
                x=m, y=max(y_opt[i] or 0, y_base[i] or 0),
                text=f'{imp:+.1f}%', showarrow=False,
                yshift=8, font=dict(size=9, color='darkgreen' if imp > 0 else 'red')
            )

    fig.update_layout(
        title=f'Baseline vs Optimized — {metric}  [K={K}, TopN={TopN}, {title_suffix}]',
        xaxis=dict(title='Method', tickangle=-35),
        yaxis=dict(title=f'Test {metric}'),
        barmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        width=1100, height=520
    )

    with out6:
        clear_output(wait=True)
        fig.show()

for w in [w6_k, w6_topn, w6_fold, w6_metric]:
    w.observe(plot_graph6, names='value')

controls6 = widgets.HBox([w6_k, w6_topn, w6_fold, w6_metric])
display(controls6, out6)
plot_graph6()

Output()

---
## 분석 노트 — Recall/Precision에서 Baseline vs Optimized 비교가 무의미한 이유

### 1. 예측 공식

본 실험의 예측값은 다음과 같이 정의된다:

$$\hat{r}(u, i) = \bar{r}_u + \alpha \cdot CF(u, i)$$

#### 각 항의 의미

**① $\bar{r}_u$ — 유저 $u$의 평균 평점**

유저가 지금까지 매긴 모든 평점의 평균값이다.

$$\bar{r}_u = \frac{1}{|\mathcal{R}_u|} \sum_{j \in \mathcal{R}_u} r(u, j)$$

- $\mathcal{R}_u$: 유저 $u$가 평점을 매긴 아이템 집합
- 예: 유저가 100개 아이템에 평균 3.8점을 줬다면 $\bar{r}_u = 3.8$
- 역할: 유저마다 다른 "평점 기준선(bias)"을 보정. 어떤 유저는 습관적으로 높은 점수, 어떤 유저는 낮은 점수를 주는 경향을 제거.

---

**② $CF(u, i)$ — KNN 협업 필터링 편차**

유저 $u$와 유사한 K명의 이웃 유저들이 아이템 $i$에 매긴 평점을, 각자의 평균에서 얼마나 벗어났는지의 가중 평균이다.

$$CF(u, i) = \frac{\displaystyle\sum_{v \in \mathcal{N}_K(u)} \mathrm{sim}(u,v) \cdot \bigl(r(v, i) - \bar{r}_v\bigr)}{\displaystyle\sum_{v \in \mathcal{N}_K(u)} |\mathrm{sim}(u,v)|}$$

- $\mathcal{N}_K(u)$: 유저 $u$와 가장 유사한 $K$명의 이웃 집합 (아이템 $i$에 평점이 있는 유저 중)
- $\mathrm{sim}(u, v)$: 본 실험의 17가지 유사도 메서드 중 하나로 계산된 유저 간 유사도
- $r(v, i) - \bar{r}_v$: 이웃 $v$가 아이템 $i$에 준 점수가 그 유저 평균 대비 얼마나 높거나 낮은지
- 결과: 양수(+)면 이웃들이 평균보다 좋아했다는 뜻, 음수(-)면 평균보다 싫어했다는 뜻

예: 이웃들이 이 아이템에 평균 대비 +0.7점을 주는 경향이 있다면 $CF(u, i) = 0.7$

---

**③ $\alpha$ — 협업 필터링 신호 조정 파라미터**

$CF(u, i)$를 예측에 얼마나 반영할지 결정하는 스케일 파라미터다.

$$\hat{r}(u, i) = \underbrace{\bar{r}_u}_{\text{유저 기준선}} + \underbrace{\alpha}_{\text{신뢰도}} \cdot \underbrace{CF(u, i)}_{\text{이웃 신호}}$$

| $\alpha$ 값 | 의미 |
|-------------|------|
| $\alpha = 0$ | 이웃 정보 완전 무시 → 모든 아이템에 $\bar{r}_u$ 예측 |
| $\alpha = 1$ | 이웃 신호를 그대로 반영 (baseline) |
| $\alpha > 1$ | 이웃 신호 증폭 (신뢰도 높음) |
| $0 < \alpha < 1$ | 이웃 신호 축소 (신뢰도 낮음, 과적합 방지) |

- **Baseline**: $\alpha = 1.0$ 고정 (CF 신호를 100% 신뢰)
- **Optimized**: validation RMSE를 최소화하는 $\alpha^*$ 탐색 (본 실험의 핵심)

---

**전체 예시**

유저 $u$의 평균 평점이 3.8, 이웃들이 아이템 $i$를 평균 대비 +0.7 좋아하고, $\alpha^* = 1.2$이면:

$$\hat{r}(u, i) = 3.8 + 1.2 \times 0.7 = 3.8 + 0.84 = 4.64$$

같은 유저의 아이템 $j$에 대해 이웃들이 −0.3이면:

$$\hat{r}(u, j) = 3.8 + 1.2 \times (-0.3) = 3.8 - 0.36 = 3.44$$

→ $\hat{r}(u, i) = 4.64 > \hat{r}(u, j) = 3.44$ 이므로 아이템 $i$가 더 높은 순위를 가짐.

---

### 2. 랭킹에서 α가 소거되는 증명

아이템 $i$와 $j$에 대한 추천 순위 비교:

$$\hat{r}(u, i) > \hat{r}(u, j)$$

$$\Leftrightarrow \quad \bar{r}_u + \alpha \cdot CF(u, i) \;\; > \;\; \bar{r}_u + \alpha \cdot CF(u, j)$$

양변에서 $\bar{r}_u$를 빼면:

$$\Leftrightarrow \quad \alpha \cdot CF(u, i) > \alpha \cdot CF(u, j)$$

$\alpha > 0$ 이면 양변을 $\alpha$로 나눌 수 있으므로:

$$\Leftrightarrow \quad CF(u, i) > CF(u, j)$$

**결론**: 아이템 간 순위는 $\alpha$값에 완전히 무관하다 ($\alpha > 0$인 한).  
따라서 TopN 추천 목록이 동일 → Recall@N, Precision@N 모두 동일 → **비교 불가**.

> **유일한 예외**: $\alpha^* \leq 0$이 되는 경우 (순위 역전 발생). 본 실험에서 alpha 탐색 범위는 양수로 제한되어 있으므로 해당 없음.

---

### 3. RMSE/MAD 비교는 왜 가능한가?

$$RMSE = \sqrt{\frac{1}{|T|}\sum_{(u,i)\in T}(\hat{r}(u,i) - r(u,i))^2}$$

$$= \sqrt{\frac{1}{|T|}\sum_{(u,i)\in T}(\bar{r}_u + \alpha \cdot CF(u,i) - r(u,i))^2}$$

α가 예측값의 **크기(scale)** 를 변경 → 실제 평점과의 오차가 달라짐 → RMSE/MAD 비교 유효.

---

### 4. Graph 6 해석

현재 Graph 6에서 baseline vs optimized의 Recall/Precision 차이가 0%로 나타나는 것은  
**실험 결과가 이상한 것이 아니라, 수학적으로 당연한 결과**이다.

---

### 5. 메서드 간 비교를 위한 실험 재설계

**현재 문제: Closed-World 평가 프로토콜**

현재 코드의 후보 아이템 선정:
```python
cand_mask = train_col.isna() & ~test_col.isna()  # test 아이템만 후보
```

유저당 평균 후보 수: **~8개** (fold 1 기준). TopN=5~10이면 후보 전부 추천 → Recall≈1.0 고착.

---

**방향 A — Open-World 프로토콜 (권장)**

후보 아이템을 전체 미평가 아이템으로 확장:
```python
cand_mask  = train_col.isna()               # 전체 미평가 ~1,587개/유저
relevant   = cand_mask & (test_col >= 4.0)  # 실제 평점 ≥ 4.0인 test 아이템만 relevant
```

| 항목 | 현재 (Closed-World) | 변경 후 (Open-World) |
|------|---------------------|----------------------|
| 후보 수/유저 | 평균 8개 | 평균 1,587개 (×144) |
| 추천 난이도 | 거의 없음 | 현실적 |
| 메서드 변별력 | 없음 | 충분 |
| RMSE 계산 | 유지 (test 아이템만) | 유지 |

기대 효과: 메서드 간 Recall@10 차이가 현재 **~0.003** → **~0.05∼0.15** 수준으로 확대.

---

**방향 B — 평점 예측 정확도 지표에 집중 (현재 설계와 정합)**

본 실험은 원래 RMSE 최소화를 목적으로 설계되었으므로,  
**랭킹 품질(Recall/Precision)은 부수 지표로만 보고**, RMSE/MAD를 주요 결과로 제시하는 방향.

이 경우 Graph 3·6은 "평점 예측 최적화가 랭킹에는 영향 없음"을 보여주는 음성 결과(negative result)로 활용 가능.

---

**방향 C — α 범위를 음수까지 확장 (실험적)**

$\alpha < 0$ 허용 시 순위 역전 발생 → baseline vs optimized Recall 차이가 생길 수 있음.  
단, RMSE 관점에서 $\alpha < 0$이 최적인 경우는 거의 없으므로 실용성 낮음.

---

**권장**: 방향 A (Open-World) + 방향 B 병행  
→ RMSE로 alpha 최적화, Recall@N은 open-world 재평가하여 메서드 변별력 확보.


---
## 재실험 설계 — Open-World Recall/Precision 평가

### 배경: 현재 실험의 한계

현재 `precision_recall_at_n()` 함수의 후보 아이템 선정:

```python
# 현재 (Closed-World)
cand_mask = train_col.isna() & ~test_col.isna()  # test 아이템만 후보
```

이 코드는 "test 세트에 평점이 존재하는 아이템"만 후보로 삼는다. 결과:

| 항목 | 수치 (fold 1 기준) |
|------|-------------------|
| 유저당 평균 후보 수 | **8개** |
| TopN=5일 때 후보 ≤ 5인 유저 비율 | **53.7%** |
| TopN=50일 때 후보 ≤ 50인 유저 비율 | **99.4%** |

→ 대부분의 유저에서 후보 전부 추천 = Recall 자동 최대 → 유사도 메서드 변별 불가.

---

### 핵심 변경: 2-Stage 평가 분리

RMSE 최적화(Phase 1)와 Recall 평가(Phase 2)를 **완전히 분리**한다.

```
Phase 1 (변경 없음)  train_inner → validation으로 alpha 최적화 (RMSE 기반)
Phase 2 (변경)       train_full  → test로 open-world Recall/Precision 평가
```

---

### Open-World 후보 선정 수식

**현재 (Closed-World)**:

$$\mathcal{C}^{CW}(u) = \{ i \mid r_{train}(u,i) = \text{NaN} \;\wedge\; r_{test}(u,i) \neq \text{NaN} \}$$

평균 $|\mathcal{C}^{CW}(u)| \approx 8$

**변경 후 (Open-World)**:

$$\mathcal{C}^{OW}(u) = \{ i \mid r_{train}(u,i) = \text{NaN} \}$$

평균 $|\mathcal{C}^{OW}(u)| \approx 1{,}587$ (전체 미평가 아이템)

**Relevant 아이템 정의는 동일**:

$$\mathcal{REL}(u) = \{ i \in \mathcal{C}^{OW}(u) \mid r_{test}(u,i) \geq 4.0 \}$$

즉, test 세트에 실제로 등장하고 평점 ≥ 4.0인 아이템만 relevant로 인정.

**Recall@N, Precision@N**:

$$\text{Recall@N}(u) = \frac{|\text{TopN}(u) \cap \mathcal{REL}(u)|}{|\mathcal{REL}(u)|}$$

$$\text{Precision@N}(u) = \frac{|\text{TopN}(u) \cap \mathcal{REL}(u)|}{N}$$

---

### 연산 구조 변경

**Alpha는 TopN과 무관**하다는 것이 검증됨 (동일 fold/method/K에서 모든 TopN에 동일한 α*). 따라서:

```
현재:  predict → evaluate (TopN loop 포함) → 34,000 runs
재설계: predict (전체 미평가 아이템) → evaluate (TopN loop 분리) → 3,400 runs
```

```
for fold in [1..10]:
    load train_full, test
    for method in [17개]:
        load sim_full_{method}.npy          # train_full 기반 sim (재사용)
        for K in [10,20,...,100]:
            for type in [baseline, optimized]:
                alpha = 1.0  또는  alpha*(fold, method, K)   ← CSV에서 조회
                
                # 한 번에 전체 미평가 아이템 예측
                pred_full = knn_predict_all_unseen(train, sim, K, alpha)
                
                # TopN은 평가 단계에서만 루프
                for TopN in [5,10,15,...,50]:
                    recall, precision = openworld_eval(pred_full, train, test, TopN)
                    results.append(...)
```

**총 예측 실행 횟수**: 17 × 10 × 10 × 2 = **3,400회** (현재 34,000의 1/10)

---

### 필요한 사전 작업: sim_full 파일 확인

현재 `sim_inner_{method}.npy`는 **train_inner** 기반. open-world 재평가는 **train_full** 기반 sim이 필요.

```
data/movielenz_data/fold_01/
├── sim_inner_cosine.npy   ← 현재 실험에서 사용 중
├── sim_full_cosine.npy    ← 필요 여부 확인 필요
└── ...
```

- `sim_full_*.npy`가 없으면 `precompute_similarity_inner.py`를 참고해 train.csv 기반으로 재계산.
- sim 행렬 크기: 943 × 943 유저 = **6.8 MB** (17개 메서드 × 10 fold × 6.8 MB = 1.2 GB, 전체 재계산 필요)

---

### 출력 스키마

**파일명**: `results/combined/openworld_recall_lambda0.csv`

| 열 | 설명 |
|----|------|
| fold | 1~10 |
| method | 17개 유사도 메서드 |
| K | 10, 20, ..., 100 |
| TopN | 5, 10, 15, ..., 50 |
| type | optimized / baseline |
| alpha | 사용된 alpha 값 |
| recall_ow | Open-World Recall@N |
| precision_ow | Open-World Precision@N |

총 행 수: 17 × 10 × 10 × 10 × 2 = **34,000행**

---

### 구현 체크리스트

- [ ] **Step 1**: `sim_full_{method}.npy` 파일 존재 여부 확인
- [ ] **Step 2**: 없으면 `precompute_sim_full.py` 작성 및 실행 (fold당 ~10분 예상)
- [ ] **Step 3**: `reeval_openworld_recall.py` 작성
  - alpha lookup: `consolidated_lambda0_final.csv`에서 (fold, method, K) → alpha* 추출
  - `precision_recall_at_n_openworld()` 함수: `cand_mask = train_col.isna()`
- [ ] **Step 4**: 실행 및 `openworld_recall_lambda0.csv` 저장
- [ ] **Step 5**: 노트북에 Graph 7 추가 (open-world 결과 시각화)

---

### 기대 효과

| 지표 | 현재 (Closed-World) | 재설계 후 (Open-World) |
|------|---------------------|------------------------|
| 메서드 간 Recall@10 범위 | ~0.003 | ~0.05~0.15 (예상) |
| cosine vs manhattan 차이 | 식별 불가 | 명확히 구별 |
| alpha 효과 (Recall) | 0% (수학적 필연) | 0% (동일, 변하지 않음) |
| 논문 비교 가능성 | 불가 | CF 논문 표준과 동일한 프로토콜 |
